# Reproducing Dehaqani, Michelon et al. (2024) from dandiset 000931

**Paper.** Dehaqani AA, Michelon F, Patella P, Petrucco L, Piasini E, Iurilli G (2024). A
mechanosensory feedback that uncouples external and self-generated sensory responses in the
olfactory cortex. *Cell Reports* 43, 114013.
[doi:10.1016/j.celrep.2024.114013](https://doi.org/10.1016/j.celrep.2024.114013).

**Text.** The claims file (`claims/000931/10.1016-j.celrep.2024.114013.yaml`) was written from the
version of record (main text, figure legends, STAR Methods and the legends of Figures S1 to S6),
and checked by a second reader. This attempt read the same text.

**Dandiset.** [DANDI:000931](https://dandiarchive.org/dandiset/000931/0.261011.1559), version
0.261011.1559 (doi:10.48324/dandi.000931/0.261011.1559). The paper cites no version of the
dandiset; its data statement says processed data "will be shared by the lead contact upon
request". The protocol then pins the most recent published version. All three published
versions and the draft hold the same single asset (same asset id and SHA-256, printed below);
only the most recent published version and the draft name the paper in their metadata. So the
choice of version does not change the data.

**What the dandiset holds.** One NWB file, `sub-001/sub-001.nwb`: the spike times of 1013 sorted
units from one mouse, recorded on 2021-10-27, with the keyword "Concentration Series". The paper
recorded 22 mice in seven groups. The file has no breathing signal or inhalation times, no odor
trials, no artificial airflow pulses, no unit waveforms and no brain areas. Every claim needs at
least one of these (printed below, per claim), so no claim can be tested.

**Claims.**

| Claim | Outcome |
|---|---|
| All 37 claims of the claims file (inhalation clustering, phase and speed decoding, odor responses and their dependence on inhalation speed, GLM and linear models, airflow tuning in anesthetized, awake, neurectomized and bulbectomized mice, motor cortex, population decoding of concentration, the simulation, and the waveform control) | Blocked: data missing |

This notebook lists what the dandiset and its file hold, checks the unit table against its
description and the paper's spike-sorting criteria, and shows, per claim, which data it needs and
which are missing.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mechano

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

started = time.time()
archive = mechano.versions()
f = mechano.open_file()
units, trains = mechano.load_units(f)
print(f"dandiset {mechano.DANDISET_ID}, pinned version {mechano.DANDISET_VERSION}; "
      f"loaded {len(units)} units and {units['count'].sum()} spikes in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Every version of the dandiset and its assets (SHA-256 truncated to 16 characters):")
print(archive.to_string(index=False))
print("\nDistinct assets over all versions:", archive.asset_id.nunique(),
      "; distinct SHA-256:", archive.sha256.nunique())
pinned = archive[archive.version == mechano.DANDISET_VERSION]
print("Pinned version:", mechano.DANDISET_VERSION, "; files:", len(pinned),
      "; subjects:", int(pinned.subjects.iloc[0]), "; variables measured:", pinned.variables.iloc[0])

The archive lists one file and one subject in every version. The three published versions
(0.240305.0939, 0.240305.0945, 0.261011.1559) and the draft share the asset, so they hold the same
data. The two 2024 versions do not name the paper among their related resources; the pinned one
does.

In [ ]:
print("Everything the file holds, outside /specifications:")
print(mechano.contents(f).to_string(index=False))
print("\nGeneral metadata:")
for key, value in mechano.general(f).items():
    print(f"  {key}: {value}")
for group in ["acquisition", "analysis", "processing", "stimulus/presentation", "stimulus/templates"]:
    print(f"/{group} holds {len(f[group])} objects")
print("/intervals present:", "intervals" in f, "; electrodes table present:",
      "general/extracellular_ephys" in f)

The file holds only general metadata and the Units table. `acquisition`, `analysis`,
`processing`, `stimulus/presentation` and `stimulus/templates` are empty; there is no `intervals`
group (no trials or epochs) and no electrodes table. The Methods describe four signals recorded
besides the spikes, all at 30 kHz: the nasal airflow from a flow sensor, the odor signal from a
photo-ionization detector, a TTL at each odor trial and a TTL at each detected inhalation onset
(STAR Methods, Experimental signals). None is in the file.

The keyword "Concentration Series" places this mouse in the group of four mice given alpha-pinene
and eucalyptol at three concentrations (STAR Methods, Mice), as far as a keyword can; the file does
not say which odors were given, and the subject age (P56D) is within the paper's 6 to 9 weeks. The
file's `related_publications` names the bioRxiv preprint (10.1101/2023.04.04.535405), not the
version of record. The session description is the converter's default ("Auto-generated by
neuroconv").

In [ ]:
import yaml

with open("../../claims/000931/10.1016-j.celrep.2024.114013.yaml") as fh:
    claim_ids = [c["id"] for c in yaml.safe_load(fh)["claims"]]
print("Claims in the claims file:", len(claim_ids), "; every one listed in mechano.CLAIM_NEEDS:",
      sorted(claim_ids) == sorted(mechano.CLAIM_NEEDS))
n_subjects = int(pinned.subjects.iloc[0])
present = mechano.streams_present(f, n_subjects)
print("Kinds of data the claims use, and whether the file holds them:")
for kind, (held, looked) in present.items():
    print(f"  {kind:15s} {'held' if held else 'NOT HELD':9s} looked for: {looked}")
    print(f"  {'':15s} {'':9s} needed: {mechano.STREAMS[kind]}")

rows = []
for claim, needs in mechano.CLAIM_NEEDS.items():
    missing = [k for k in needs if not present[k][0]]
    rows.append({"claim": claim, "needs": ", ".join(needs), "missing": ", ".join(missing),
                 "testable": not missing})
needs_table = pd.DataFrame(rows)
print("\nPer claim:")
print(needs_table.to_string(index=False))
print("\nClaims:", len(needs_table), "; testable on this file:", int(needs_table.testable.sum()))
print("Claims missing the breathing signal or inhalation times:",
      int(needs_table.missing.str.contains("breathing").sum()))
print("Claims missing odor trial times:", int(needs_table.missing.str.contains("odor_trials").sum()))
print("Claims missing artificial airflow pulses:", int(needs_table.missing.str.contains("airflow_pulses").sum()))
print("Claims missing unit waveforms:", int(needs_table.missing.str.contains("waveforms").sum()))

Each claim's needs are read from its criterion in the claims file and the STAR Methods sections it
cites (the docstring of `mechano.py` says so). No claim can be tested: 33 need the breathing
signal or inhalation times, 31 need odor trial times, 2 need the artificial airflow pulses of the
tracheostomized mice, and 4 need unit waveforms. Beyond that, every claim pools mice of one or
more groups (the four concentration-series mice, the six of Figure 1, the ten control, four
bulbectomized, five neurectomized or three anesthetized mice), and most select PCx layer 2 and 3
units, for which the file has no brain area. The spikes of one mouse alone cannot stand in for
any of them, since the paper's every measure on spikes is aligned to an inhalation, an odor
onset or an airflow pulse. No outside source holds these data publicly: the paper says the
processed data are available on request, and its code deposit (Zenodo
10.5281/zenodo.10381293, read by the reader at commit 3bec502) loads preprocessed .mat files it
does not contain.

With no trial table or event times in the file, there is no second stream whose clock could be
checked against the spikes (protocol, Author, step 2); that check could not be made.

In [ ]:
duration = float(units.last_spike.max())
print("Units:", len(units), "; unit_name unique:", units.unit_name.nunique(),
      "; original_cluster_id unique:", units.original_cluster_id.nunique())
print("\nKilosort label (KSLabel) against Phy curation (quality):")
print(pd.crosstab(units.quality, units.KSLabel, margins=True).to_string())
print("\nContamPct values:", units.ContamPct.value_counts().to_dict())
print("sh values:", units.sh.value_counts().to_dict())
print("\nSpike times (s): first", round(units.first_spike.min(), 4), "; last", round(duration, 4),
      f"({duration / 3600:.2f} h); every train sorted:", bool(units.sorted.all()))
print("n_spikes equals the number of spike times for every unit:", bool((units.n_spikes == units["count"]).all()))
ratio = units["count"] / duration / units.fr
print("count / last spike time / fr: min", round(ratio.min(), 6), "max", round(ratio.max(), 6),
      "; implied duration behind fr (s):", round(float((units["count"] / units.fr).median()), 2))
print("\nRanges:")
for col, unit in [("fr", "Hz"), ("depth", "um (assumed)"), ("ch", "channel"), ("Amplitude", "none given"),
                  ("amp", "none given"), ("n_spikes", "spikes")]:
    print(f"  {col:10s} {units[col].min():10.3f} to {units[col].max():10.3f}  [{unit}]")
good = units.quality == "good"
print("\nUnits curated good in Phy:", int(good.sum()),
      f"; of these, at or above {mechano.MIN_RATE_HZ} Hz:", int((good & (units.fr >= mechano.MIN_RATE_HZ)).sum()),
      "; lowest rate among them:", round(units.fr[good].min(), 3), "Hz")
print(f"All units at or above {mechano.MIN_RATE_HZ} Hz:", int((units.fr >= mechano.MIN_RATE_HZ).sum()))
print("\nPhy quality by depth (um):")
print(pd.crosstab(units.quality, pd.cut(units.depth, [0, 1000, 2000, 3000, 4000])).to_string())
print("\nPaper: 464 neurons in the four concentration-series mice (claims file, fig2c-interaction-small), "
      "116 per mouse on average, from PCx layers 2 and 3.")

Checks of the unit table against its description and the paper:

- `n_spikes` equals the number of spike times of every unit, and every spike train is sorted.
- `fr` is each unit's spike count divided by 7677.39 s, a little longer than the last spike
  (7676.04 s), so it is the mean rate over the whole recording, consistent with its name; the
  table does not describe it.
- `KSLabel` is "good" for all 1013 units and `ContamPct` is 100 for all of them. In Kilosort a
  cluster is labelled good when its estimated contamination is low, so these two columns
  contradict each other and carry no information about the units. Both are undescribed.
- `quality` (Phy curation) is "good" for 359 units, "mua" for 10, and missing for 644. The paper
  analyses single units after manual curation in Phy; the 359 good units all fire at or above
  0.5 Hz (lowest 0.515 Hz), so the paper's rate threshold removes none of them, while 629 units of
  any label pass it. Which units the paper kept is not stated in the file.
- `sh` is 0 for every unit (one shank, as a Neuropixels 1.0 probe has). `ch` runs from 6 to 382
  and `depth` from 80 to 3840, the length of the probe's recording sites in micrometres, though no
  unit is given. Good units are found along the whole probe, so they come from PCx and from the
  structures above it; the paper keeps only PCx layer 2 and 3 units (464 in four mice, 116 per
  mouse on average), and the file gives no area to select them by.
- `Amplitude` and `amp` are two amplitude measures with no description or unit.

In [ ]:
good_idx = np.where(good)[0]
isi = np.array([mechano.isi_violation_rate(trains[i], duration) for i in good_idx])
presence = np.array([mechano.presence_ratio(trains[i], duration) for i in good_idx])
q = lambda x: f"median {np.median(x):.4f}, Q1 {np.percentile(x, 25):.4f}, Q3 {np.percentile(x, 75):.4f}, max {x.max():.4f}"
print(f"Phy-good units ({len(good_idx)}):")
print("  ISI violation rate (Hill et al. 2011, 1.5 ms):", q(isi))
print("  units with an ISI violation rate >= 0.5 (the paper's exclusion threshold):", int((isi >= 0.5).sum()))
print("  presence ratio (100 blocks):", q(presence))
print("Paper (STAR Methods, Spike sorting): ISI violation rate threshold < 0.5, median 0.01, Q1 0, Q3 0.04;"
      " presence ratio median 1, Q1 1, Q3 1.")

The ISI violation rate of the good units (median 0.0066, Q1 0, Q3 0.029, none at or above 0.5) and
their presence ratio (1 in every quartile) are consistent with the ranges the Methods report for
the paper's units (median 0.01, Q1 0, Q3 0.04; presence ratio 1). The Methods do not give the
formula or the refractory period, so this comparison uses a standard one (Hill et al. 2011, 1.5 ms)
and is a consistency check only.

In [ ]:
identical, pairs = mechano.duplicates(units, trains)
print("Units with a spike train identical to another unit's:", identical)
print(f"Pairs within {mechano.NEAR_CHANNELS} channels compared:", len(pairs))
print(f"Pairs where more than 10% of the smaller unit's spikes coincide within "
      f"{mechano.COINCIDENCE_S * 1000:.1f} ms:", int((pairs.coincident > 0.1).sum()))
top = pairs.sort_values("coincident", ascending=False).head(5).copy()
top["quality_a"] = units.quality.loc[top.unit_a].to_numpy()
top["quality_b"] = units.quality.loc[top.unit_b].to_numpy()
print(top.round(3).to_string(index=False))
both_good = units.quality.loc[pairs.unit_a].to_numpy() == "good"
both_good &= units.quality.loc[pairs.unit_b].to_numpy() == "good"
print("Pairs of two Phy-good units:", int(both_good.sum()), "; largest coincident fraction among them:",
      round(float(pairs.coincident[both_good].max()), 3))

No two units have identical spike trains. Of the pairs of units within four channels, two share
more than 10% of the smaller unit's spikes within 0.2 ms; both pair unit 695 (136 spikes) with
unit 672 or 668, and Phy left all three unlabelled. Among the 1749 pairs of two good units the largest
shared fraction is 0.026. The file holds
one recording, so no recording is deposited twice.

In [ ]:
bands = [(0, 1000), (1000, 2000), (2000, 3000), (3000, 4000)]
fig, ax = plt.subplots(figsize=(7, 3.5))
rows = []
for lo, hi in bands:
    sel = np.where(good & (units.depth > lo) & (units.depth <= hi))[0]
    freq, power = mechano.rate_spectrum([trains[i] for i in sel], duration)
    base = np.median(power[(freq >= 15) & (freq <= 40)])
    band = (freq >= 1) & (freq <= 15)
    rows.append({"depth_um": f"{lo}-{hi}", "units": len(sel), "peak_hz_1_15": round(freq[band][np.argmax(power[band])], 2),
                 "peak_over_15_40Hz_median": round(power[band].max() / base, 1)})
    ax.semilogy(freq, power / base, label=f"{lo}-{hi} um ({len(sel)} units)")
ax.set_xlim(0, 20)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("power / median power 15-40 Hz")
ax.legend(fontsize=8)
plt.show()
print(pd.DataFrame(rows).to_string(index=False))
shown = freq <= 20
print("Plotted range of power / baseline over 0-20 Hz is printed per band above as the peak; frequency axis 0-20 Hz,",
      "x values beyond it not shown:", int((~shown).sum()), "frequencies.")

The Methods say the probe was lowered to "a region with high firing rates and breathing-coupled
activity" (Stereotaxic probe insertion). Without a breathing signal this cannot be shown, but the
summed activity of the 97 good units with `depth` at most 1000 (in Kilosort's convention, the end
nearest the probe tip, where the PCx would be) has its spectral peak between
1 and 15 Hz at 2.78 Hz, 9.7 times the 15 to 40 Hz baseline, a frequency within the range of
resting breathing in mice. The bands further from the tip peak near 1 Hz or have a weaker peak.
This is consistent with these units being in a breathing-coupled region such as the PCx,
and does not show that they are.

## Claims

No claim has a section of its own: none can be run on this file. The table printed under "Data
check" gives, for each of the 37 claims, the data its criterion needs and which of them are
missing; it is the output each blocked outcome in the attempt record points to (cell tag
`blocked-claims`).

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")

## Summary

| Claim | Outcome | Missing from the dandiset |
|---|---|---|
| inhalation-sorting | Blocked: data missing | breathing signal; the other awake mice |
| figs1a-phase-decoding | Blocked: data missing | breathing signal, brain areas, other mice |
| figs1b-speed-decoding-single-mice | Blocked: data missing | breathing signal, odor trials, brain areas |
| fig1-odor-responsive-fraction | Blocked: data missing | odor trials, brain areas, the other five mice |
| fig1hi-sniff-changes-responses, fig1h-direction-of-change, figs1c-early-70ms, fig1j-sniff-change-larger-than-concentration, fig1l-cdi-mixed-lower | Blocked: data missing | breathing signal, odor trials, brain areas, other mice |
| figs1d-rs-fs-similar | Blocked: data missing | as above, and unit waveforms |
| fig2-glm-variance, fig2c-interaction-small, fig2d-mixed-sensitivities, figs2d-70ms-similar | Blocked: data missing | breathing signal, odor trials, brain areas, the other three concentration-series mice |
| figs2ab-rs-fs-heterogeneous, figs2c-waveform-no-prediction | Blocked: data missing | as above, and unit waveforms |
| fig3c-airflow-tuning-anesthetized, fig3b-latency-shortens | Blocked: data missing | the anesthetized mice and their airflow pulses |
| fig3f-airflow-tuning-awake, figs3-aen-neurectomy | Blocked: data missing | breathing signal, the control and neurectomized mice |
| figs4-bulbectomy-no-odor-no-phase, fig4-sniff-preferring-fraction, fig4e-sniff-response-smaller, fig4f-sniff-response-more-variable, fig4g-sniff-decoding-reduced, fig4d-pre-sniff-rise, figs4-motor-cortex-sniff | Blocked: data missing | breathing signal, odor trials, brain areas, the control and bulbectomized mice |
| fig5b-pc1-speed-pc23-concentration, fig5c-encoding-angles, fig5d-sniff-invariant-decoding, figs5gh-other-windows-and-codes, fig5ef-correlated-subpopulations, fig5gh-mechanosensitive-as-good, fig5i-time-course-similar, figs5i-coefficients-similar, fig6-simulation-mechanosensory-rescues | Blocked: data missing | breathing signal, odor trials, brain areas, the other concentration-series and identity mice |
| figs6h-waveforms-homogeneous | Blocked: data missing | unit waveforms, the other groups of mice |

**How the analysis differs from the paper's.** No analysis of the paper was run. The notebook
checks the file against its description and against the spike-sorting criteria of the Methods.

**What was found about the dataset.**

- The dandiset holds one NWB file from one of the paper's 22 mice: spike times of 1013 units
  and their Kilosort and Phy labels. The breathing signal, the inhalation and odor-trial TTLs,
  the odor signal, the artificial airflow pulses, unit waveforms and brain areas are not in it,
  and neither are the other 21 mice. None of the paper's findings can be tested from it.
- The file does not say which experiment it belongs to except through the keyword
  "Concentration Series", which odors were given, or which units the paper analysed.
- `KSLabel` is "good" and `ContamPct` is 100 for every unit, which contradict each other;
  `quality` is missing for 644 of 1013 units. The unit table's columns other than `quality`,
  `spike_times` and `unit_name` have no description, and `depth`, `Amplitude` and `amp` no unit.
- The file's `related_publications` names the preprint, not the published paper; the two 2024
  versions of the dandiset do not name the paper.
- What could be checked is consistent with the paper: the good units' ISI violation rates and
  presence ratios fall in the ranges the Methods report, no unit is duplicated, and the deepest
  units carry a rhythm at 2.78 Hz, within the range of mouse breathing.